<a href="https://www.kaggle.com/code/byydbck/llmresearch1?scriptVersionId=363766650" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install unsloth #optimizes the LLM training execution. this project is so much faster with unsloth.
!pip install --no-deps --upgrade "torchao>=0.16.0" #optimizes tensor operations,reduce memory footprint
!pip install transformers==4.57.6 #for tokenization
!pip install --no-deps trl==0.22.2 #transformer reinforcement learning 
#used --no-deps for trl and torchao because we didn't want them to automatically update the dependencies for version errors.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 134.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 23.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 118.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 80.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.3 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0━━━  0/11 [torchao]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [unsloth]0/11 [unsloth]zoo]]ropy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 84.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from unsloth import FastLanguageModel
import torch

"""loaded the qwen2.5-7B model, assigned 2048 as the maximum token context length for one sequence
enabled 4-bit quantization and automatically detected optimal data type
"""
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B",
    max_seq_length=2048, 
    dtype=None,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.10: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
FastLanguageModel.for_inference(model) #switched the model to inference mode for faster generation


#defined the standart alpaca prompt format
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""



#tokenized the prompt and moved tensors to the CUDA GPU
FastLanguageModel.for_inference(model) 
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonacci sequence.", #instruction
        "1, 1, 2, 3, 5, 8", #input
        "", #output is empty for the model to fill
    )
], return_tensors = "pt").to("cuda") #return as python tensors


#decode back to token ids -> strings
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
outputs = model.generate(
    **inputs,         
    max_new_tokens=64, 
    use_cache=True,     
    streamer=text_streamer #streams the responses in real-time instead of making you wait without outcome
)
tokenizer.batch_decode(outputs)
#these codes were taken from the Unsloth Qwen2.5 7B Kaggle documentation. 
#https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen2_(7B)-Alpaca.ipynb#scrollTo=_g5uNxjbNx_2


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Continue the fibonacci sequence.

### Input:
1, 1, 2, 3, 5, 8

### Response:
13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6


['Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nContinue the fibonacci sequence.\n\n### Input:\n1, 1, 2, 3, 5, 8\n\n### Response:\n13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6']

In [4]:
from datasets import load_dataset
dataset = load_dataset("yahma/alpaca-cleaned", split="train") #imported the dataset from hugging face using datasets lib

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [5]:
print(dataset[0])
#as we can see there are outputs,inputs and instructions. not all data need inputs, so for it to be blank is considered normal.

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [6]:
test_dataset = dataset.shuffle(seed=42).select(range(300))
print(f"test set examples: {len(test_dataset)}")
print({test_dataset[0]['instruction']})
#"all assignment options require at least 200 held-out test examples." we will be using the same test set for each evaluation (for B1,B2 and B3)

test set examples: 300
{'Rearrange the following sentence to make the sentence more interesting.'}


In [8]:
import pandas as pd

shot_1 = dataset[500]
shot_2 = dataset[501]

# for B1: zero-shot 
samples = 200
b1_results = []

for i in range(samples):
    sample = test_dataset[i]
    instruction = sample["instruction"]
    input_text = sample["input"]
    output_text = sample["output"]
    
    prompt_b1 = alpaca_prompt.format(instruction, input_text, "")
    inputs_b1 = tokenizer([prompt_b1], return_tensors="pt").to("cuda")
    
    outputs_b1 = model.generate(
        **inputs_b1, 
        max_new_tokens=256, 
        use_cache=True,
        do_sample=False
    )
    
    # extract only the newly generated tokens (this part of the code is AI generated)
    input_len = inputs_b1["input_ids"].shape[1]
    pred_b1 = tokenizer.decode(outputs_b1[0][input_len:], skip_special_tokens=True).strip()
    
    b1_results.append({
        "instruction": instruction,
        "input_text": input_text,
        "output_text": output_text,
        "pred_b1_zero_shot": pred_b1
    })


In [10]:
df_b1 = pd.DataFrame(b1_results)
df_b1.to_csv("b1_predictions.csv", index=False)

pd.set_option('display.max_colwidth', None)
df_b1.head(5)

,instruction,input_text,output_text,pred_b1_zero_shot
0,Rearrange the following sentence to make the sentence more interesting.,She left the party early,"Early, she left the party.","She left the party early, but not before she had a great time dancing and chatting with her friends."
1,"Let \n f(x) = {[ -x - 3 if x ≤ 1,; x/2 + 1 if x > 1. ].\nFind the sum of all values of x such that f(x) = 0.",,"We solve the equation f(x) = 0 on the domains x ≤ 1 and x > 1.\n\nIf x ≤ 1, then f(x) = -x - 3, so we want to solve -x - 3 = 0. The solution is x = -3, which satisfies x ≤ 1.\n\nIf x > 1, then f(x) = x/2 + 1, so we want to solve x/2 + 1 = 0. The solution is x = -2, but this value does not satisfy x > 1.\n\nTherefore, the only solution is x = -3.","To find the sum of all values of x such that f(x) = 0, we need to solve the equation f(x) = 0 for both cases of the piecewise function.\n\nCase 1: x ≤ 1\nf(x) = -x - 3\nSetting f(x) = 0, we get:\n-x - 3 = 0\nSolving for x, we get:\nx = -3\n\nCase 2: x > 1\nf(x) = x/2 + 1\nSetting f(x) = 0, we get:\nx/2 + 1 = 0\nSolving for x, we get:\nx = -2\n\nTherefore, the sum of all values of x such that f(x) = 0 is -3 + (-2) = -5."
2,Compose a haiku poem about a summer day.,,"Warm breeze on my face\nEndless sun brings joy and peace\nSummer, please don't fade","A warm summer day,\nGolden sun shining bright,\nNature's beauty awakes."
3,What methods can be used to improve the accuracy of machine learning models?,,"Here are several methods to improve the accuracy of machine learning models:\n\n1. Gathering and cleaning more data: The more data the model is trained on, the more patterns it can recognize, and the better it can generalize to new data. It’s also important to clean the data to handle missing, inconsistent or duplicate information.\n\n2. Data preprocessing: Several preprocessing techniques like normalization, standardization, encoding, and feature selection can improve model accuracy by making the data more suitable for the learning algorithm.\n\n3. Selecting the right algorithm: Different algorithms are suited to different types of data and problems. Experimenting with various algorithms, including ensemble methods that combine several algorithms, can boost accuracy.\n\n4. Hyperparameter tuning: Hyperparameters are the fixed parameters that control the behavior of the learning algorithm. Tuning them can sometimes result in large improvements in accuracy.\n\n5. Regularization: Regularization is a technique to prevent overfitting by adding a penalty term to the loss function. It encourages the model to use all the input features in a more balanced way, thus resulting in better generalization on new data.\n\n6. Cross-validation: Cross-validation is a technique to evaluate how well the model generalizes on new, unseen data. By using cross-validation to help choose the best combination of preprocessing techniques, algorithms, and hyperparameters, it’s possible to increase model accuracy.","There are several methods that can be used to improve the accuracy of machine learning models, including:\n\n1. Feature engineering: This involves selecting and transforming the input features to make them more relevant and informative for the model. This can be done by removing irrelevant features, scaling or normalizing the features, or creating new features that capture important patterns in the data.\n\n2. Hyperparameter tuning: This involves adjusting the parameters of the model to optimize its performance. This can be done using techniques such as grid search, random search, or Bayesian optimization.\n\n3. Ensemble methods: This involves combining multiple models to improve their accuracy. This can be done using techniques such as bagging, boosting, or stacking.\n\n4. Regularization: This involves adding a penalty term to the loss function to prevent overfitting. This can be done using techniques such as L1 or L2 regularization, dropout, or early stopping.\n\n5. Data augmentation: This involves generating new

In [ ]:
import pandas as pd

#for B2: chain-of-thought. we are doing almost the same thing for b2. we add a prompt for llm to process everything "step by step"
samples = 200
b2_results = []

for i in range(samples):
    sample = test_dataset[i]
    instruction = sample["instruction"]
    input_text = sample["input"]
    output_text = sample["output"]
    # CoT prompting: instruct the model to break down the task step by step
    cot_instruction = instruction + " Let's think step by step to arrive at the correct answer."
    
    prompt_b2 = alpaca_prompt.format(cot_instruction, input_text, "")
    inputs_b2 = tokenizer([prompt_b2], return_tensors="pt").to("cuda")
    
    outputs_b2 = model.generate(
        **inputs_b2, 
        max_new_tokens=256,  
        use_cache=True,
        do_sample=False
    )
    # extract only the newly generated tokens
    input_len = inputs_b2["input_ids"].shape[1]
    pred_b2 = tokenizer.decode(outputs_b2[0][input_len:], skip_special_tokens=True).strip()
    # show as a dataframe 
    b2_results.append({
        "instruction": instruction,
        "input_text": input_text,
        "output_text": output_text,
        "pred_b2_cot": pred_b2
    })


In [ ]:
df_b2 = pd.DataFrame(b2_results)
df_b2.to_csv("b2_predictions.csv", index=False)

pd.set_option('display.max_colwidth', None)
df_b2.head(5)

In [ ]:
# for B3: PAL (Program-Aided Language Model)
samples = 200
b3_results = []

for i in range(samples):
    sample = test_dataset[i]
    instruction = sample["instruction"]
    input_text = sample["input"]
    output_text = sample["output"]
    
    #PAL prompting: Instruct the model to write Python code instead of plain text
    pal_instruction = f"Solve the following problem by writing a Python script. Write the executable Python code that computes the result. Then, simulate running the code and clearly state the final printed output.\n\nTask:{instruction}"
    
    prompt_b3 = alpaca_prompt.format(pal_instruction, input_text, "")
    inputs_b3 = tokenizer([prompt_b3], return_tensors="pt").to("cuda")
    
    outputs_b3 = model.generate(
        **inputs_b3, 
        max_new_tokens=256, 
        use_cache=True,
        do_sample=False
    )
    
    # extract only the newly generated tokens
    input_len = inputs_b3["input_ids"].shape[1]
    pred_b3 = tokenizer.decode(outputs_b3[0][input_len:], skip_special_tokens=True).strip()
    
    b3_results.append({
        "instruction": instruction,
        "input_text": input_text,
        "output_text": output_text,
        "pred_b3_pal": pred_b3
    })


In [ ]:
df_b3 = pd.DataFrame(b3_results)
df_b3.to_csv("b3_predictions.csv", index=False)

pd.set_option('display.max_colwidth', None)
df_b3.head(5)

In [ ]:
!pip install -q evaluate rouge_score bert_score

import evaluate

# load metrics
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')

# true outputs (same for all)
references = df_b1["output_text"].tolist()

# calculate b1 metrics
rouge_b1 = rouge.compute(predictions=df_b1["pred_b1_zero_shot"].tolist(), references=references)
bert_b1 = bertscore.compute(predictions=df_b1["pred_b1_zero_shot"].tolist(), references=references, lang="en")
bert_f1_b1 = sum(bert_b1['f1']) / len(bert_b1['f1'])

# calculate b2 metrics
rouge_b2 = rouge.compute(predictions=df_b2["pred_b2_cot"].tolist(), references=references)
bert_b2 = bertscore.compute(predictions=df_b2["pred_b2_cot"].tolist(), references=references, lang="en")
bert_f1_b2 = sum(bert_b2['f1']) / len(bert_b2['f1'])

# calculate b3 metrics
rouge_b3 = rouge.compute(predictions=df_b3["pred_b3_pal"].tolist(), references=references)
bert_b3 = bertscore.compute(predictions=df_b3["pred_b3_pal"].tolist(), references=references, lang="en")
bert_f1_b3 = sum(bert_b3['f1']) / len(bert_b3['f1'])

# print final scores for the report screenshot
print("--- b1 zero-shot scores ---")
print(f"rouge-l: {rouge_b1['rougeL']:.4f}")
print(f"bertscore f1: {bert_f1_b1:.4f}\n")

print("--- b2 cot scores ---")
print(f"rouge-l: {rouge_b2['rougeL']:.4f}")
print(f"bertscore f1: {bert_f1_b2:.4f}\n")

print("--- b3 pal scores ---")
print(f"rouge-l: {rouge_b3['rougeL']:.4f}")
print(f"bertscore f1: {bert_f1_b3:.4f}")

In [ ]:
import evaluate

# load bleu metric 
bleu = evaluate.load('bleu')

# true outputs
references = df_b1["output_text"].tolist()

# calculate bleu for all conditions
bleu_b1 = bleu.compute(predictions=df_b1["pred_b1_zero_shot"].tolist(), references=references)
bleu_b2 = bleu.compute(predictions=df_b2["pred_b2_cot"].tolist(), references=references)
bleu_b3 = bleu.compute(predictions=df_b3["pred_b3_pal"].tolist(), references=references)

# print bleu scores to compare efficiency
print("--- bleu scores (for paper discussion) ---")
print(f"b1 zero-shot: {bleu_b1['bleu']:.4f}")
print(f"b2 cot: {bleu_b2['bleu']:.4f}")
print(f"b3 pal: {bleu_b3['bleu']:.4f}")